In [1]:
import os

import numpy as np
import pandas as pd

# Fresh weights

### 2020 Summer

In [2]:
input_df = pd.read_csv('./results/2020_S/SW2_greenhouse.csv', index_col='Unnamed: 0').interpolate()
input_df.index = pd.DatetimeIndex(input_df.index)

In [3]:
input_df = input_df.resample('10 min').mean()

In [4]:
output_df = pd.read_csv('./results/2020_S/fw_labels.csv', index_col='Unnamed: 0')

In [5]:
input_data_2020S_CT = [pd.concat([input_df.loc[:, :'permit'], input_df.loc[:, 'loadcell_1']], axis=1).values,
                       pd.concat([input_df.loc[:, :'permit'], input_df.loc[:, 'loadcell_2']], axis=1).values,
                       pd.concat([input_df.loc[:, :'permit'], input_df.loc[:, 'loadcell_3']], axis=1).values]

In [6]:
data_indices_2020S_CT = [np.array(output_df['CT_1'].index),
                         np.array(output_df['CT_2'].index),
                         np.array(output_df['CT_3'].index)]

In [7]:
output_label_2020S_CT = [output_df['CT_1'].values,
                         output_df['CT_2'].values,
                         output_df['CT_3'].values]

In [8]:
input_df = pd.read_csv('./results/2020_S/SW2_FR_greenhouse.csv', index_col='Unnamed: 0').interpolate()

In [9]:
output_df = pd.read_csv('./results/2020_S/fw_labels.csv', index_col='Unnamed: 0')

In [10]:
input_data = np.concatenate(input_data_2020S_CT)
data_indices = np.concatenate(data_indices_2020S_CT).reshape(-1, 1)
output_label = np.concatenate(output_label_2020S_CT).reshape(-1, 1)

In [11]:
INPUT_MAXS = input_data.reshape(-1, 9).max(axis=0)
INPUT_MINS = input_data.reshape(-1, 9).min(axis=0)
OUTPUT_MAX = output_label.max()
OUTPUT_MIN = output_label.min()

In [12]:
input_data = (input_data - INPUT_MINS)/(INPUT_MAXS - INPUT_MINS)
input_data = input_data.reshape(-1, 144, 9)
output_label = (output_label - OUTPUT_MIN)/(OUTPUT_MAX - OUTPUT_MIN)

In [13]:
f = open('./results/2020_S/fw_dataset.npz', 'wb')
np.savez(f,
         data_indices = data_indices,
         input_data = input_data,
         output_label = output_label,
         INPUT_MAXS = INPUT_MAXS,
         INPUT_MINS = INPUT_MINS,
         OUTPUT_MAX = OUTPUT_MAX,
         OUTPUT_MIN = OUTPUT_MIN
        )
f.close()

In [15]:
print(input_data.shape)
print(output_label.shape)
print(data_indices.shape)

(363, 144, 9)
(363, 1)
(363, 1)


### 2020 Winter

In [ ]:
input_df = pd.read_csv('./results/2020_W/SW2_greenhouse.csv', index_col='Unnamed: 0').interpolate()

In [ ]:
output_df = pd.read_csv('./results/2020_W/fw_labels.csv', index_col='Unnamed: 0')

In [ ]:
input_data_2020W_CT = [pd.concat([input_df.loc[:, :'permit'], input_df.loc[:, 'loadcell_1']], axis=1).values,
                       pd.concat([input_df.loc[:, :'permit'], input_df.loc[:, 'loadcell_2']], axis=1).values,
                       pd.concat([input_df.loc[:, :'permit'], input_df.loc[:, 'loadcell_3']], axis=1).values]

In [ ]:
data_indices_2020W_CT = [np.array(output_df['CT_1'].index),
                         np.array(output_df['CT_2'].index),
                         np.array(output_df['CT_3'].index)]

In [ ]:
output_label_2020W_CT = [output_df['CT_1'].values,
                         output_df['CT_2'].values,
                         output_df['CT_3'].values]

### 2020 Winter

In [ ]:
DIRECTORY = './data/2020_W/'
file_list = os.listdir(DIRECTORY)
dataset_list = [file for file in file_list if file.endswith('.dat')]
dataset_list.sort()

In [ ]:
dataset_list

In [ ]:
greenhouse_df = []
for FILENAME in dataset_list:
    try:
        temp_df = pd.read_csv(DIRECTORY + FILENAME, sep='\t', index_col='TIMESTAMP', skiprows=[0, 2, 3])
    except ValueError:
        temp_df = pd.read_csv(DIRECTORY + FILENAME, sep=',', index_col='TIMESTAMP', skiprows=[0, 2, 3])
    greenhouse_df.append(temp_df)
greenhouse_df = pd.concat(greenhouse_df, sort=True)

In [ ]:
greenhouse_df.index = pd.DatetimeIndex(greenhouse_df.index)

In [ ]:
cr1000_df = greenhouse_df
cr1000_df = cr1000_df[['Loadcell_1', 'Loadcell_2', 'Pyrano_Wsec_1', 'Temp_Avg', 'Humid_Avg']]

#### date range

In [ ]:
cr1000_df = cr1000_df.loc['2020-08-26 00:00:00':]

In [ ]:
date_index = pd.date_range(cr1000_df.index[0], cr1000_df.index[-1], freq='1 min')

In [ ]:
cr1000_df = cr1000_df.reindex(date_index)

In [ ]:
DIRECTORY = './data/2020_W/'
file_list = os.listdir(DIRECTORY)
dataset_list = [file for file in file_list if file.endswith('.xlsx')]
dataset_list.sort()

In [ ]:
dataset_list

In [ ]:
greenhouse_df = []
div_1 = 0
div_2 = 0
div_3 = 0
for FILENAME in dataset_list:
    if 'CT' in FILENAME:
        div_1 += 1
    if 'FR' in FILENAME:
        div_2 += 1
    if 'RB1' in FILENAME:
        continue
    if 'RB2' in FILENAME:
        div_3 += 1
    temp_df = pd.read_excel(DIRECTORY + FILENAME, index_col='date')
    temp_df.index = pd.DatetimeIndex(temp_df.index)
    temp_df = temp_df[['weight', 'subs_VWC', 'subs_EC', 'subs_bulk_EC', 'subs_temp', 'permittivity']]
    greenhouse_df.append(temp_df)

In [ ]:
other_df = []
for FILENAME in dataset_list:
    if 'RB1' in FILENAME:
        temp_df = pd.read_excel(DIRECTORY + FILENAME, index_col='date')
        temp_df.index = pd.DatetimeIndex(temp_df.index)
        temp_df = temp_df[['weight']]
        other_df.append(temp_df)
    else:
        continue

In [ ]:
for i in range(len(greenhouse_df)):
    if greenhouse_df[i].index.round('1 min')[0].minute == 1:
        greenhouse_df[i].index = (greenhouse_df[i].index.round('1 min') - pd.Timedelta('1 min'))
    elif greenhouse_df[i].index.round('1 min')[0].minute == 2:
        greenhouse_df[i].index = (greenhouse_df[i].index.round('2 min') - pd.Timedelta('2 min'))
    else:
        greenhouse_df[i].index = greenhouse_df[i].index.round('1 min')

In [ ]:
for i in range(len(greenhouse_df)):
    greenhouse_df[i] = greenhouse_df[i].groupby(greenhouse_df[i].index).first()

In [ ]:
date_index = pd.date_range(greenhouse_df[0].index[0], greenhouse_df[-1].index[-1], freq='1 min')

In [ ]:
_1 = pd.concat(greenhouse_df[:div_1]).reindex(date_index)
_2 = pd.concat(greenhouse_df[div_1:div_1+div_2]).reindex(date_index)
_3 = pd.concat(greenhouse_df[div_1+div_2:]).reindex(date_index)

In [ ]:
_1.columns = ['Loadcell_3', 'subs_VWC', 'subs_EC', 'subs_bulk_EC', 'subs_temp', 'permittivity']
_2.columns = ['Loadcell_4', 'subs_VWC', 'subs_EC', 'subs_bulk_EC', 'subs_temp', 'permittivity']
_3.columns = ['Loadcell_5', 'subs_VWC', 'subs_EC', 'subs_bulk_EC', 'subs_temp', 'permittivity']

In [ ]:
SW2_df = pd.concat([_1.loc[pd.date_range(cr1000_df.index[0], _1.index[-1], freq='1 min')],
                    cr1000_df.loc[pd.date_range(cr1000_df.index[0], _1.index[-1], freq='1 min')]], axis=1)

In [ ]:
SW2_df.columns = ['loadcell_3', 'subs_VWC', 'subs_EC', 'subs_bulk_EC', 'subs_temp', 'permit', 'loadcell_1', 'loadcell_2', 'rad', 'temp', 'hum']

In [ ]:
SW2_df = SW2_df[['temp', 'hum', 'rad', 'subs_VWC', 'subs_EC', 'subs_bulk_EC', 'subs_temp', 'permit', 'loadcell_1', 'loadcell_2', 'loadcell_3']]

In [ ]:
SW2_df['temp'] = ((SW2_df['temp']-1000)/80)
SW2_df['hum'] = ((SW2_df['hum']-1000)/40)
SW2_df['loadcell_1'] = ((SW2_df['loadcell_1']/100))
SW2_df['loadcell_2'] = ((SW2_df['loadcell_2']/100))

In [ ]:
SW2_df.to_csv('./results/2020_W/SW_CT_greenhouse.csv')